In [ ]:
import deepinv as dinv
import torch
import matplotlib.pyplot as plt
from deepinv.utils.demo import load_dataset
from torchvision import transforms
import numpy as np
from sapg import SAPG

from priors import WaveletPrior, GSDPrior, L2, CombinedPrior, L1Prior, CRRPrior
from utils import plot_MC_corr, device
from models.utils import load_model

### Forward model, noise

In [ ]:
# Set the global random seed from pytorch to ensure reproducibility of the example.
torch.manual_seed(0)

# Set up the variable to fetch dataset and operators.
dataset_name = "set3c"
img_size = 128
n_channels = 1

natural_img = True

if natural_img:
    val_transform = transforms.Compose([transforms.CenterCrop(img_size), transforms.ToTensor()])

    dataset = load_dataset(dataset_name, transform=val_transform)
    im_ind = 0
    x = dataset[im_ind][0].unsqueeze(0).to(device)
    
    if n_channels == 1:
        deco = dinv.physics.Decolorize(device=device)
        x = deco(x)

else:
    sigma_img = 0.25
    np.random.seed(1)
    x = torch.tensor(np.random.laplace(0., sigma_img, [1, 1, img_size, img_size]).astype(np.float32)).to(device)

In [ ]:
dimx = x.shape[-1]*x.shape[-2]

filter_torch = dinv.physics.blur.gaussian_blur(sigma=(2, 2))
noise_level_img = 0.05  # Gaussian Noise standard deviation for the degradation

# The BlurFFT instance from physics enables to compute efficently backward operators with Fourier transform.
p = dinv.physics.Blur(
    img_size=(1, img_size, img_size),
    filter=filter_torch,
    device=device, padding="circular",
    noise_model=dinv.physics.GaussianNoise(sigma=noise_level_img))

# generate blurred, noisy measurement
y = p(x) 

# plot the signal, the measurement and the noise
dinv.utils.plot([x, y, y - x], ["signal", "measurement", "diff"], rescale_mode='clip')

In [ ]:
# Lipschitz constant of nabla f
L_f = p.compute_norm(x0=torch.randn_like(x), tol=1e-5) / noise_level_img**2
f = L2(noise_level_img, p)

# regularization parameter of proximity operator (lambda)
lam_reg = min(1/L_f, 2.)   

## Test 1 : wavelets

In [ ]:
theta0 = torch.tensor(20.)

# Lipschitz constant of grad g
L_g =  1/lam_reg 

# Lipschitz constant of grad g + f
L =  L_f + L_g

# Stepsize of MCMC algorithm
gamma = 0.98*1/L
print("gamma: {}  lam: {}".format(gamma, lam_reg))
g = WaveletPrior(theta0)
f = L2(noise_level_img, p)

d0 = 50 / img_size**2 / theta0 
print("d0 = ", d0)
delta = lambda k: d0 / (k ** 0.4)

## Test 2: L1

In [ ]:
theta0 = torch.tensor(1.)

L_g =  1/lam_reg 
L =  L_f + L_g

gamma = 0.98*1/L
print("gamma: {}  lam: {}".format(gamma, lam_reg))
g = L1Prior(theta0)

d0 = 20 / img_size**2 / theta0 
print("d0 = ", d0)
delta = lambda k: d0 / (k ** 0.5)

## Test 3 : Gradient Step Denoiser explicit prior

In [ ]:
path_ckpt = "GSDRUNet_grayscale_torch.ckpt" if n_channels == 1 else "GSDRUNet_torch.ckpt"
denoiser = dinv.models.GSDRUNet(pretrained=path_ckpt, 
                                in_channels=1, out_channels=1, device=device)
    
theta0 = torch.tensor(150.)

# Lipschitz constant of grad g
L_g = 2

# Lipschitz constant of grad g + f
L =  L_f + L_g

# Stepsize of MCMC algorithm
gamma = 0.98*1/L
print("gamma: {}  lam: {}".format(gamma, lam_reg))
g = GSDPrior(theta0, denoiser)

d0 = 75. / img_size**2 / theta0 #20. / img_size**2 / theta0 
print("d0 = ", d0)
delta = lambda k: d0 / (k ** 0.4)

## Test 4 : Convex Ridge Regularizer

In [ ]:
theta0 = torch.tensor((25., 4.), device=device)  # lambda regularization parameter, mu scaling parameter
crr_model = load_model("../convex_ridge_regularizers/trained_models/Sigma_25_t_10/checkpoints/checkpoint_10.pth")
crr_model.initializeEigen(size=200)

g = CRRPrior(theta0, crr_model)

L_g = crr_model.precise_lipschitz_bound(n_iter=500)*theta0[1]#*theta0[0]
L =  L_f + L_g
gamma = 0.98*1/L
d0 = 10. / img_size**2 / theta0 #20. / img_size**2 / theta0 
print("d0 = ", d0)
delta = lambda k: d0 / (k ** 0.4)

## Combined prior

In [ ]:
p1 = GSDPrior(torch.tensor(106., device=device), denoiser) #  132.8
p2 = L1Prior(torch.tensor(4., device=device))
theta0 = torch.tensor(0.5)
g = CombinedPrior(torch.tensor(theta0, device=device), p1, p2)

d0 = 250. / img_size**2 / theta0 #20. / img_size**2 / theta0 
print("d0 = ", d0)
delta = lambda k: d0 / (k ** 0.1)

## Warming up the MC sampler

In [ ]:
g.update_param(theta0)
sapg = SAPG(y, g, f, gamma, 0.98*gamma, lam_reg, X_init=p.A_A_adjoint(y).to(device).detach().clone(), project=None) # gamma instead of 0.98 gamma
nwarm = 20000
plot_stats = True
warm_prior = False
thinning_prior = 20

if warm_prior:
    res = sapg.warm_up(nwarm, log_stats=plot_stats, burnin_ratio=0.95, warm_up_prior=True, thinning_prior=thinning_prior)
else:
    res = sapg.warm_up(nwarm, log_stats=plot_stats, burnin_ratio=0.95, warm_up_prior=False)
if plot_stats:
    if warm_prior:
        X_post, X_prior, post, prior = res
    else: 
        X_post, post = res
    plt.plot(post, label='post')
    if warm_prior:
        X_prior, prior = res_prior
        plt.plot(prior, label='prior')
    plt.legend();

In [ ]:
if plot_stats:
    print(X_post.shape)    
    start_ind, thinning = 0, 1
    print((len(X_post) - start_ind) // thinning)
    fig, ax = plt.subplots(3, 3, figsize=(13, 7))
    plot_MC_corr(X_post[start_ind::thinning, :].numpy(), ax=ax[0], title='Xpost')
    if warm_prior:
        plot_MC_corr(X_prior[start_ind::thinning, :].numpy(), ax=ax[1], title='Xprior')
        plot_MC_corr(prior[start_ind::thinning].numpy().reshape([-1, 1]), ax=ax[2], title='g(Xprior)')
    plt.tight_layout()

In [ ]:
if plot_stats:
    xmean = torch.mean(X_post.reshape([-1, img_size, img_size]), axis=0)
    dinv.utils.plot([xmean.reshape([1, img_size, img_size]), x, y])
    psnr = dinv.loss.metric.PSNR()
    print("PSNR : ", psnr(xmean.reshape([1, 1, img_size, img_size]), x.cpu()).item())

In [ ]:
param, param_hist, mean_hist, post_hist, prior_hist = sapg.run(delta, 1000, (1., 1000), init_param=theta0, reuse_post=True,  # (0.0001, 1000)
                                                               thinning_post=100, thinning_prior=200, alpha=None, thinning_global=1, tol=1e-6) # 100, 200


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 5))
ax[0].plot(-post_hist.numpy(), label=r'$g(X)$')
ax[0].plot(-prior_hist.numpy(), label=r'$g(\bar X)$')
ax[1].plot(param_hist.numpy(), label=r'$\theta$')
ax[2].plot(mean_hist.numpy(), label=r'$\bar{\theta}$')
ax[0].legend(), ax[1].legend(), ax[2].legend();
#ax[0].set_yscale('log')
plt.tight_layout()
plt.savefig("figs/plot_sapg_crr.pdf")

In [ ]:
X_hist, post_hist = sapg.sample_post(20000) 
print(X_hist[-1].shape)
mean_img = torch.tensor(torch.mean(X_hist[10000::10], axis=0), device=device)
dinv.utils.plot([mean_img, y, (x - mean_img)], ["reconstr", "measurement", "diff"], rescale_mode='clip')
psnr = dinv.loss.metric.PSNR()
print("PSNR : ", psnr(mean_img, x).item())